In [1]:
!pip install --quiet pandas numpy scikit-learn joblib nltk textblob vaderSentiment matplotlib seaborn streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 27.9 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import nltk, re, joblib
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# Download NLTK resources
nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
vader = SentimentIntensityAnalyzer()


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [3]:
# Upload or mount the Kaggle dataset (stock_sentiment.csv)
df = pd.read_csv("stock_sentiment.csv.zip")

print("Shape:", df.shape)
print(df.head())

# Rename columns (if needed)
df = df.rename(columns={'Text':'content', 'Sentiment':'label'})


Shape: (5791, 2)
                                                Text  Sentiment
0  Kickers on my watchlist XIDE TIT SOQ PNK CPW B...          1
1  user: AAP MOVIE. 55% return for the FEA/GEED i...          1
2  user I'd be afraid to short AMZN - they are lo...          1
3                                  MNTA Over 12.00            1
4                                   OI  Over 21.37            1


In [4]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = text.split()
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return " ".join(tokens)

def compute_sentiment_features(df):
    vader_scores, tb_polarity, tb_subjectivity = [], [], []
    for txt in df['content'].astype(str):
        vs = vader.polarity_scores(txt)
        vader_scores.append(vs['compound'])
        tb = TextBlob(txt)
        tb_polarity.append(tb.sentiment.polarity)
        tb_subjectivity.append(tb.sentiment.subjectivity)
    df['vader_compound'] = vader_scores
    df['tb_polarity'] = tb_polarity
    df['tb_subjectivity'] = tb_subjectivity
    return df

# Apply preprocessing
df['clean'] = df['content'].apply(clean_text)
df = compute_sentiment_features(df)


In [5]:
from scipy.sparse import hstack

X_text = df['clean'].values
X_num = df[['vader_compound','tb_polarity','tb_subjectivity']].values
y = df['label'].values

tfidf = TfidfVectorizer(max_features=6000, ngram_range=(1,2))
X_text_tfidf = tfidf.fit_transform(X_text)
X = hstack([X_text_tfidf, X_num])


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Model + Hyperparameter tuning
lr = LogisticRegression(max_iter=1000, class_weight='balanced', solver='saga')
param_grid = {'C':[0.1, 1, 5, 10]}
grid = GridSearchCV(lr, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_


In [7]:
y_pred = best_model.predict(X_test)
print("✅ Accuracy:", round(accuracy_score(y_test, y_pred)*100, 2), "%\n")
print(classification_report(y_test, y_pred))


✅ Accuracy: 80.59 %

              precision    recall  f1-score   support

          -1       0.74      0.71      0.73       421
           1       0.84      0.86      0.85       738

    accuracy                           0.81      1159
   macro avg       0.79      0.79      0.79      1159
weighted avg       0.80      0.81      0.81      1159



In [35]:
print(df.columns)

Index(['content', 'label', 'clean', 'vader_compound', 'tb_polarity',
       'tb_subjectivity'],
      dtype='object')


In [36]:
print(df['label'].unique())

[ 1 -1]


In [8]:
joblib.dump(tfidf, "tfidf.joblib")
joblib.dump(best_model, "model.joblib")
print("✅ Model and TF-IDF saved successfully.")


✅ Model and TF-IDF saved successfully.


In [37]:
%%writefile app.py

import streamlit as st
import joblib
import re
import numpy as np
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from scipy.sparse import hstack

# Load trained model & vectorizer
model = joblib.load("model.joblib")
tfidf = joblib.load("tfidf.joblib")

# Initialize VADER
vader = SentimentIntensityAnalyzer()

# Text cleaning function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    return text

# Streamlit page config
st.set_page_config(
    page_title="StockSentinel",
    layout="centered"
)

# Title
st.title("📈 StockSentinel – AI Powered Market Sentiment Analyzer")

# Text input
text_input = st.text_area(
    "Enter Tweet / Stock-related text:",
    height=150
)

# Button
if st.button("Analyze Sentiment"):

    # Empty input check
    if text_input.strip() == "":
        st.warning("Please enter some text!")

    else:

        # Clean text
        clean = clean_text(text_input)

        # VADER sentiment
        vs = vader.polarity_scores(text_input)

        # TextBlob sentiment
        tb = TextBlob(text_input)

        # TF-IDF transform
        X_text = tfidf.transform([clean])

        # Numerical features
        X_num = np.array([[
            vs['compound'],
            tb.sentiment.polarity,
            tb.sentiment.subjectivity
        ]])

        # Combine features
        X_final = hstack([X_text, X_num])

        # Prediction
        pred = model.predict(X_final)[0]

        # Label mapping
        label_map = {
           -1: "Negative",
            1: "Positive"
        }

        # Convert prediction to label
        pred_label = label_map.get(int(pred), str(pred))

        # Show result
        st.success(
            f"Predicted Sentiment: **{pred_label.upper()}** 🧠"
        )

        # Additional score display
        st.subheader("📊 Sentiment Scores")

        st.write(f"VADER Compound Score: {vs['compound']}")
        st.write(f"TextBlob Polarity: {tb.sentiment.polarity}")
        st.write(f"TextBlob Subjectivity: {tb.sentiment.subjectivity}")

Overwriting app.py


In [38]:
!pip install streamlit pyngrok
!pip install joblib scikit-learn nltk


In [39]:
from pyngrok import ngrok

# Replace with your own ngrok token
ngrok.set_auth_token("31ViNQHcXZQxmzvoUfoOD8xUagW_7ybpdjTkdHvhAK4JNrojN")


In [40]:
!pkill -f ngrok


In [41]:
!streamlit run app.py &




2026-05-22 11:38:42.925 Uvicorn server started on 0.0.0.0:8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://34.139.169.91:8502

  Stopping...


In [42]:
from pyngrok import ngrok

# Connect ngrok to Streamlit port (8501)
public_url = ngrok.connect(8501)
print("Your public app URL:", public_url)


Your public app URL: NgrokTunnel: "https://3b12-34-139-169-91.ngrok-free.app" -> "http://localhost:8501"


In [43]:
#  Install
!pip install streamlit pyngrok -q

#  Import and start Streamlit properly
from pyngrok import ngrok
import threading
import time
import subprocess

# Start the Streamlit app in background
def run_streamlit():
    process = subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501"])
    process.wait()

thread = threading.Thread(target=run_streamlit)
thread.start()

# Wait few seconds to make sure Streamlit starts
time.sleep(10)

#  Create public URL
public_url = ngrok.connect(8501)
print("✅ Your Streamlit App is Live at:", public_url.public_url)


✅ Your Streamlit App is Live at: https://0c89-34-139-169-91.ngrok-free.app
